In [1]:
import os
import numpy as np
import pandas as pd
import torch

from sklearn.decomposition import PCA

import matplotlib.pyplot as plt
import seaborn as sns

In [2]:
from cue.core.model import CUE, CUEConfigs

In [3]:
import warnings
warnings.filterwarnings("ignore", category=UserWarning)

In [4]:
random_state = 2025

In [5]:
sns.set_theme(style='darkgrid')

# 1. Configuration

## Task Configuration

In [6]:
# The shared ChEMBL tables keep the original assay folder names and column spellings;
# everything this project reads and writes uses the unified names (DIS/INH, WS2/RS2).
DATASET_SRC = {'DIS': 'Kd', 'INH': 'Ki'}
LEGACY_COLUMNS = {'WS(alpha=2)': 'WS2', 'WS(alpha=1)': 'WS1',
                  'RS(k=2)': 'RS2', 'RS(k=3)': 'RS3'}


def normalize_columns(df):
    hits = {c: LEGACY_COLUMNS[c] for c in df.columns if c in LEGACY_COLUMNS}
    return df.rename(columns=hits) if hits else df

DATASETS = ['DIS', 'INH']

TARGET_COLS = [
    'PI',
    'Ssel',
    'WS2',
    'RS2',
    'S(1uM)_label',
    'S(100nM)_label'
]

START_FOLD = 1
END_FOLD = 5
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

In [7]:
label_map = {
    'PI': 'regression',
    'Ssel': 'regression',
    'RS2': 'regression',
    'WS2': 'regression',
    # 'S(1uM)_label': 'classification',
    # 'S(100nM)_label': 'classification',
}

In [8]:
def doPCA(tr_embs, va_embs, te_embs, tr_y, va_y, te_y, target_col, random_state):
    pca = PCA(n_components=2, random_state=random_state)
    pca.fit(tr_embs)

    tr_points = pca.transform(tr_embs)
    va_points = pca.transform(va_embs)
    te_points = pca.transform(te_embs)

    df_tr = pd.DataFrame({
        "PC1": tr_points[:, 0],
        "PC2": tr_points[:, 1],
        target_col: tr_y
    })

    df_va = pd.DataFrame({
        "PC1": va_points[:, 0],
        "PC2": va_points[:, 1],
        target_col: va_y
    })

    df_te = pd.DataFrame({
        "PC1": te_points[:, 0],
        "PC2": te_points[:, 1],
        target_col: te_y
    })

    x_min, x_max = tr_points[:, 0].min(), tr_points[:, 0].max()
    y_min, y_max = tr_points[:, 1].min(), tr_points[:, 1].max()

    x_pad = (x_max - x_min) * 0.05
    y_pad = (y_max - y_min) * 0.05

    axis_range = {
        "x_min": x_min - x_pad,
        "x_max": x_max + x_pad,
        "y_min": y_min - y_pad,
        "y_max": y_max + y_pad,
    }

    return df_tr, df_va, df_te, axis_range

In [9]:
def scatterplot(df, filepath, target_col, target_type, axis_range, title='Embedding Distribution'):
    """target_col 값을 색상으로 사용한 scatter plot"""
    fig, ax = plt.subplots(1, 1, figsize=(8, 6))
    
    if target_type == 'regression':
        # Regression: 기존처럼 rainbow colormap 사용
        scatter = ax.scatter(df['PC1'], df['PC2'], c=df[target_col], cmap='rainbow', alpha=0.8, s=20)
        
        cbar = plt.colorbar(scatter, ax=ax)
        # cbar.set_label(target_col, fontsize=12)
        
    else:
        # Classification: 0=Blue, 1=Red 로 이진 분류 시각화
        colors = {0: 'blue', 1: 'red'}
        
        # 각 클래스별로 scatter plot을 그려서 legend 생성
        unique_labels = sorted(df[target_col].unique())
        for label in unique_labels:
            subset = df[df[target_col] == label]
            color_val = colors.get(int(label), 'gray') # 0, 1 이외의 값이면 회색
            ax.scatter(subset['PC1'], subset['PC2'], c=color_val, label=f'Class {int(label)}', alpha=0.8, s=20)
            
        # ax.legend() # 범례 추가

    ax.set_xlim(axis_range['x_min'], axis_range['x_max'])
    ax.set_ylim(axis_range['y_min'], axis_range['y_max'])

    # 공통 축 라벨 및 타이틀 설정
    # ax.set_xlabel('PC1', fontsize=15)
    # ax.set_ylabel('PC2', fontsize=15)
    # ax.set_title(title, fontsize=16)
    
    # plt.tight_layout()
    plt.savefig(filepath, dpi=300, bbox_inches='tight')
    plt.close()

In [10]:
import json
import os
import numpy as np
import pandas as pd

for DATASET in DATASETS:
    for TARGET_COL in TARGET_COLS:
        TASK_TYPE = label_map.get(TARGET_COL, 'regression')
        VERSION = "v1"
        
        # config.json 경로는 fold 폴더의 상위 폴더
        model_root = f'./results/{DATASET}/{TARGET_COL}/{VERSION}/CUE'
        config_path = os.path.join(model_root, 'config.json')

        if not os.path.exists(config_path):
            print(f"[SKIP] Config not found: {config_path}")
            continue

        with open(config_path, 'r') as f:
            train_cfg = json.load(f)

        print("\n" + "=" * 80)
        print(f"[INFO] Start: DATASET={DATASET} | TARGET_COL={TARGET_COL} | TASK_TYPE={TASK_TYPE}")
        print("=" * 80)

        for fold_num in range(START_FOLD, END_FOLD + 1):
            print(f"\n[Fold {fold_num}] Processing...")
            
            results_dir = os.path.join(model_root, f'fold{fold_num}')
            # the shared ChEMBL tables keep the on-disk assay folder names (DIS -> Kd)
            data_dir = f'./../chembl/data/selectivity_df/{DATASET_SRC[DATASET]}/fold{fold_num}'

            # 데이터 로드
            df_tr = normalize_columns(pd.read_csv(os.path.join(data_dir, 'selectivity_train.csv')))
            df_va = normalize_columns(pd.read_csv(os.path.join(data_dir, 'selectivity_val.csv')))
            df_te = normalize_columns(pd.read_csv(os.path.join(data_dir, 'selectivity_test.csv')))

            # Fusion 가중치 로드
            fusion_dir = os.path.join(results_dir, 'fusion')
            f_tr = pd.read_csv(os.path.join(fusion_dir, 'train_fusion.csv'))
            f_va = pd.read_csv(os.path.join(fusion_dir, 'val_fusion.csv'))
            f_te = pd.read_csv(os.path.join(fusion_dir, 'test_fusion.csv'))

            # 가중치 매칭
            df_tr = pd.merge(df_tr, f_tr[['compound_chembl_id', 'w_fp', 'w_img']], on='compound_chembl_id', how='left')
            df_va = pd.merge(df_va, f_va[['compound_chembl_id', 'w_fp', 'w_img']], on='compound_chembl_id', how='left')
            df_te = pd.merge(df_te, f_te[['compound_chembl_id', 'w_fp', 'w_img']], on='compound_chembl_id', how='left')

            X_tr, y_tr, w_tr = df_tr['SMILES'].values, df_tr[TARGET_COL].values.astype(np.float32), df_tr[['w_fp', 'w_img']].values
            X_va, y_va, w_va = df_va['SMILES'].values, df_va[TARGET_COL].values.astype(np.float32), df_va[['w_fp', 'w_img']].values
            X_te, y_te, w_te = df_te['SMILES'].values, df_te[TARGET_COL].values.astype(np.float32), df_te[['w_fp', 'w_img']].values

            # 모델 설정 (Config 값 반영)
            cfg = CUEConfigs(
                fold_dir=results_dir,
                device=DEVICE,
                hidden_dim=train_cfg.get('hidden_dim', 32),
                task_type=train_cfg.get('task_type', 'regression')
            )
            model = CUE(cfg)
            model.load()

            # 가중치 기반 예측
            predict_params = {
                "fusion_policy": train_cfg.get('fusion_policy', 'softmax'),
                "temperature": train_cfg.get('temperature', 1.0),
                "uncertainty_norm": train_cfg.get('uncertainty_norm', 'none')
            }
            
            emb_tr = model.predict(X_tr, weights=w_tr, **predict_params)
            emb_va = model.predict(X_va, weights=w_va, **predict_params)
            emb_te = model.predict(X_te, weights=w_te, **predict_params)

            # PCA 수행
            df_tr_pca, df_va_pca, df_te_pca, axis_range = doPCA(
                emb_tr, emb_va, emb_te,
                y_tr, y_va, y_te,
                target_col=TARGET_COL,
                random_state=random_state
            )

            # 결과 저장
            output_dir = os.path.join(results_dir, 'embeddings')
            os.makedirs(output_dir, exist_ok=True)
            figure_dir = os.path.join(output_dir, 'PCA')
            os.makedirs(figure_dir, exist_ok=True)

            scatterplot(df_tr_pca, os.path.join(figure_dir, 'train_embeddings.png'), TARGET_COL, TASK_TYPE, axis_range, title=f'Train PCA (Fold {fold_num})')
            scatterplot(df_va_pca, os.path.join(figure_dir, 'val_embeddings.png'), TARGET_COL, TASK_TYPE, axis_range, title=f'Val PCA (Fold {fold_num})')
            scatterplot(df_te_pca, os.path.join(figure_dir, 'test_embeddings.png'), TARGET_COL, TASK_TYPE, axis_range, title=f'Test PCA (Fold {fold_num})')

            np.save(os.path.join(output_dir, 'train_embeddings.npy'), {'embeddings': emb_tr, TARGET_COL: y_tr, 'SMILES': X_tr})
            np.save(os.path.join(output_dir, 'val_embeddings.npy'), {'embeddings': emb_va, TARGET_COL: y_va, 'SMILES': X_va})
            np.save(os.path.join(output_dir, 'test_embeddings.npy'), {'embeddings': emb_te, TARGET_COL: y_te, 'SMILES': X_te})

            print(f"[Fold {fold_num}] Completed!")


[INFO] Start: DATASET=Kd | TARGET_COL=PI | TASK_TYPE=regression

[Fold 1] Processing...
[Fold 1] Completed!

[Fold 2] Processing...
[Fold 2] Completed!

[Fold 3] Processing...
[Fold 3] Completed!

[Fold 4] Processing...
[Fold 4] Completed!

[Fold 5] Processing...
[Fold 5] Completed!

[INFO] Start: DATASET=Kd | TARGET_COL=Ssel | TASK_TYPE=regression

[Fold 1] Processing...
[Fold 1] Completed!

[Fold 2] Processing...
[Fold 2] Completed!

[Fold 3] Processing...
[Fold 3] Completed!

[Fold 4] Processing...
[Fold 4] Completed!

[Fold 5] Processing...
[Fold 5] Completed!

[INFO] Start: DATASET=Kd | TARGET_COL=WS(alpha=2) | TASK_TYPE=regression

[Fold 1] Processing...
[Fold 1] Completed!

[Fold 2] Processing...
[Fold 2] Completed!

[Fold 3] Processing...
[Fold 3] Completed!

[Fold 4] Processing...
[Fold 4] Completed!

[Fold 5] Processing...
[Fold 5] Completed!

[INFO] Start: DATASET=Kd | TARGET_COL=RS(k=2) | TASK_TYPE=regression

[Fold 1] Processing...
[Fold 1] Completed!

[Fold 2] Processing